# Principio de Sustitución de Liskov (LSP)

Una subclase debe sustituir al tipo base sin romper su contrato.

## Sin aplicar LSP

El padre promete rastreo, pero `EntregaRecogidaLocal` lanza una excepción inesperada. El cliente genérico deja de funcionar.

In [ ]:
class Entrega:
    def __init__(self, codigo, direccion):
        self.codigo = codigo; self.direccion = direccion; self.estado = 'creada'
    def programar(self, hora):
        self.estado = f'programada para {hora}'; return self.estado
    def rastrear(self): return f'{self.codigo}: {self.estado}'

class EntregaRecogidaLocal(Entrega):
    def __init__(self, codigo, direccion):
        super().__init__(codigo, direccion); self.punto_recogida = direccion; self.requiere_transporte = False
    def programar(self, hora):
        self.estado = f'lista desde {hora}'; return self.estado
    def rastrear(self): raise RuntimeError('Una recogida local no tiene rastreo')

def consultar(entrega):
    entrega.programar('18:00'); return entrega.rastrear()

for item in (Entrega('DOM-01', 'Calle 10'), EntregaRecogidaLocal('LOC-01', 'Sede Centro')):
    try: print(consultar(item))
    except RuntimeError as error: print(f'Contrato roto para {item.codigo}: {error}')

## Aplicando LSP

El contrato incluye solo entregas que garantizan programación y rastreo. Domicilio y bicicleta conservan las mismas condiciones y tipos de retorno.

In [ ]:
from abc import ABC, abstractmethod

class EntregaRastreable(ABC):
    def __init__(self, codigo, destino):
        self.codigo = codigo; self.destino = destino; self.estado = 'creada'
    @abstractmethod
    def programar(self, hora): pass
    @abstractmethod
    def rastrear(self): pass

class EntregaDomicilio(EntregaRastreable):
    def __init__(self, codigo, destino):
        super().__init__(codigo, destino); self.transporte = 'Moto'; self.hora = None
    def programar(self, hora): self.hora = hora; self.estado = 'en preparación'; return self.estado
    def rastrear(self): return f'{self.codigo} a {self.destino}: {self.estado}, {self.transporte}'

class EntregaBicicleta(EntregaRastreable):
    def __init__(self, codigo, destino):
        super().__init__(codigo, destino); self.transporte = 'Bicicleta'; self.hora = None
    def programar(self, hora): self.hora = hora; self.estado = 'en preparación'; return self.estado
    def rastrear(self): return f'{self.codigo} a {self.destino}: {self.estado}, {self.transporte}'

class MonitorEntregas:
    def __init__(self, nombre, entregas=None):
        self.nombre = nombre; self.entregas = entregas if entregas is not None else []
    def agregar(self, entrega): self.entregas.append(entrega); return len(self.entregas)
    def consultar_todas(self, hora):
        for entrega in self.entregas: entrega.programar(hora)
        return [entrega.rastrear() for entrega in self.entregas]

monitor = MonitorEntregas('Noche', [EntregaDomicilio('DOM-02','Carrera 20'), EntregaBicicleta('BICI-01','Calle 5')])
for resultado in monitor.consultar_todas('19:00'): print(resultado)

El monitor usa ambas subclases sin condicionales ni excepciones especiales: cualquiera sustituye correctamente al contrato base.